# pyINLA Interactive Demo

Welcome to **pyINLA** - Fast Bayesian inference for Latent Gaussian Models in Python.

This notebook lets you try pyINLA directly in your browser using Google Colab.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyinla/pyinla/blob/main/demo.ipynb)

## 1. Installation

First, install pyINLA and download the INLA computational engine.

In [ ]:
!pip install pyinla -q

In [ ]:
import pyinla

# Download the INLA binary (required on first run)
pyinla.download_binary()

## 2. Simple Linear Regression

Let's start with a basic example: predicting house prices based on size.

We'll:
1. Simulate some data
2. Fit a Bayesian linear regression
3. Examine the posterior distributions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyinla import pyinla

# Set seed for reproducibility
np.random.seed(42)

# Simulate house price data
n = 100
size = np.random.uniform(50, 200, n)  # House size in m²
true_intercept = 50000
true_slope = 2000  # Price per m²
noise = np.random.normal(0, 20000, n)

price = true_intercept + true_slope * size + noise

# Create DataFrame
df = pd.DataFrame({'price': price, 'size': size})
print(df.head())

In [ ]:
# Define the model
model = {
    'response': 'price',
    'fixed': ['1', 'size']  # Intercept + size effect
}

# Fit the model
result = pyinla(model, family='gaussian', data=df)

# View fixed effects summary
print("Fixed Effects:")
print(result.summary_fixed)

In [ ]:
# Visualize the posterior for the slope (size effect)
from pyinla import dmarginal

marg_size = result.marginals_fixed['size']

plt.figure(figsize=(8, 4))
plt.plot(marg_size['x'], marg_size['y'], 'b-', linewidth=2)
plt.axvline(true_slope, color='r', linestyle='--', label=f'True value: {true_slope}')
plt.xlabel('Price per m²')
plt.ylabel('Posterior density')
plt.title('Posterior Distribution of Size Effect')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Poisson Regression (Count Data)

Now let's try a different likelihood: Poisson regression for count data.

Example: Modeling the number of website visits based on advertising spend.

In [ ]:
# Simulate count data
np.random.seed(123)
n = 80

ad_spend = np.random.uniform(1, 10, n)  # Advertising spend (thousands)
log_rate = 2 + 0.3 * ad_spend  # True model: log(visits) = 2 + 0.3 * ad_spend
visits = np.random.poisson(np.exp(log_rate))

df_counts = pd.DataFrame({'visits': visits, 'ad_spend': ad_spend})

# Plot the data
plt.figure(figsize=(8, 4))
plt.scatter(ad_spend, visits, alpha=0.6)
plt.xlabel('Ad Spend (thousands)')
plt.ylabel('Website Visits')
plt.title('Visits vs Advertising Spend')
plt.show()

In [ ]:
# Fit Poisson regression
model_poisson = {
    'response': 'visits',
    'fixed': ['1', 'ad_spend']
}

result_poisson = pyinla(model_poisson, family='poisson', data=df_counts)

print("Poisson Regression Results:")
print(result_poisson.summary_fixed)
print("\nTrue values: intercept=2.0, ad_spend=0.3")

## 4. Random Effects (Mixed Models)

pyINLA supports random effects for hierarchical/multilevel models.

Example: Student test scores with school-level random effects.

In [ ]:
# Simulate hierarchical data
np.random.seed(456)

n_schools = 10
students_per_school = 20
n_total = n_schools * students_per_school

# School random effects
school_effects = np.random.normal(0, 10, n_schools)

# Generate data
school_id = np.repeat(range(1, n_schools + 1), students_per_school)
hours_studied = np.random.uniform(1, 10, n_total)

# True model: score = 50 + 5*hours + school_effect + noise
score = (50 + 5 * hours_studied + 
         np.array([school_effects[s-1] for s in school_id]) + 
         np.random.normal(0, 5, n_total))

df_schools = pd.DataFrame({
    'score': score,
    'hours': hours_studied,
    'school': school_id
})

print(df_schools.head(10))

In [ ]:
# Fit mixed effects model
model_mixed = {
    'response': 'score',
    'fixed': ['1', 'hours'],
    'random': {
        'school': {'model': 'iid'}  # IID random intercepts for schools
    }
}

result_mixed = pyinla(model_mixed, family='gaussian', data=df_schools)

print("Fixed Effects:")
print(result_mixed.summary_fixed)
print("\nHyperparameters (includes random effect variance):")
print(result_mixed.summary_hyperpar)

## 5. What's Next?

This demo covered the basics. pyINLA supports much more:

- **20+ likelihood families**: Gaussian, Poisson, Binomial, Gamma, Beta, survival models, etc.
- **Spatial models**: SPDE for geostatistics, Besag/BYM2 for areal data
- **Time series**: Random walks (RW1, RW2), AR models, seasonal effects
- **Model comparison**: DIC, WAIC, CPO for model selection

### Learn More

- **Documentation**: [pyinla.org/docs](https://pyinla.org/docs)
- **Examples**: [pyinla.org/docs/examples](https://pyinla.org/docs/examples)
- **Applications**: [pyinla.org/apps](https://pyinla.org/apps)
- **GitHub**: [github.com/pyinla/pyinla](https://github.com/pyinla/pyinla)

In [ ]:
# Your turn! Try modifying the examples above or create your own model.
# 
# Tips:
# - Change 'family' to try different likelihoods: 'binomial', 'gamma', 'nbinomial'
# - Add more predictors to the 'fixed' list
# - Explore the result object: result.dic, result.cpo, result.marginals_hyperpar